# Tesco Clubcard: Nutrient Diversity in London

This notebook contains the analysis for an MSc Data Science project exploring nutrient diversity in grocery purchasing across London, including statistical and geospatial analysis.

## Before running

1. Obtain the source datasets from their original publishers and check their reuse conditions.
2. Place the required files in `data/raw/` at the repository root:
   - `year_msoa_grocery.csv`
   - `Net_income_bhc_equivalised.csv`
   - `MSOA_Dec_2011_Boundaries_Generalised_Clipped_BGC_EW_V3_2022_-5730664396045573288.geojson`
3. Install the dependencies listed in `requirements.txt`.
4. Run the notebook from the repository root or from the `notebooks/` directory.

The source datasets are not included in this repository. The income dataset's exact source, release and licence, and the boundary file's source and licence, still need confirmation.

This is a cleaned portfolio copy. The original academic notebook should be retained separately.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import sklearn
import geopandas as gpd
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [ ]:
# Load source datasets using repository-relative paths
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

grocery_path = DATA_DIR / "year_msoa_grocery.csv"
income_path = DATA_DIR / "Net_income_bhc_equivalised.csv"

if not grocery_path.exists() or not income_path.exists():
    raise FileNotFoundError(
        "Required CSV files were not found. Place year_msoa_grocery.csv and "
        "Net_income_bhc_equivalised.csv in the repository's data/raw/ folder. "
        "See data/README.md for source and licensing notes."
    )

df_msoa_grocery = pd.read_csv(grocery_path)
df_ni_bhc = pd.read_csv(income_path, encoding="latin1")

print(df_msoa_grocery.columns)
print()
df_ni_bhc.info()


In [ ]:
# Merge datasets

# change variable name to area_id before merging
df_ni_bhc.rename(columns={'MSOA code':'area_id'},inplace=True)

# Left join msoa grocery data with income data on area_id

df_msoa_grocery_income = df_msoa_grocery.merge(df_ni_bhc,how='left',on='area_id')
df_msoa_grocery_income.head()


### Insight 2: What drives Nutrient Diversity
This section examines the relationship between nutrient diversity and the diversity of protein, carbohydrate and sugar purchasing patterns. Shannon entropy is calculated separately for each macronutrient group, and OLS regression is used to investigate their associations with overall nutrient diversity.

In [ ]:
# Calculate normalized columns and add them to the original DataFrame
df_msoa_grocery_income['f_fruit_veg_norm'] = (df_msoa_grocery_income['f_fruit_veg']-df_msoa_grocery_income['f_fruit_veg'].min())/(df_msoa_grocery_income['f_fruit_veg'].max()-df_msoa_grocery_income['f_fruit_veg'].min())
df_msoa_grocery_income['f_readymade_norm'] = (df_msoa_grocery_income['f_readymade']-df_msoa_grocery_income['f_readymade'].min())/(df_msoa_grocery_income['f_readymade'].max()-df_msoa_grocery_income['f_readymade'].min())

df_pro_carb_div = df_msoa_grocery_income[['area_id',
                                         'f_meat_red',
                                         'f_poultry',
                                         'f_dairy',
                                         'f_eggs',
                                         'f_fish',
                                         'f_grains',
                                         'f_sweets',
                                         'f_soft_drinks',
                                         'f_readymade_weight',
                                         'f_readymade',
                                         'f_fruit_veg',
                                         'f_fruit_veg_weight',
                                         'h_nutrients_weight',
                                         'h_nutrients_weight_norm',
                                         'f_fruit_veg_norm',
                                         'f_readymade_norm',
                                         'avg_age',
                                         'energy_density'
                                         ]].copy()

protein = ['f_meat_red', 'f_poultry','f_eggs','f_dairy','f_fish']
carbs = ['f_grains','f_fruit_veg','f_readymade']
sugar = ['f_sweets','f_soft_drinks']

df_pro_carb_div['protein_Sp'] = df_pro_carb_div[protein].sum(axis=1)
df_pro_carb_div['carb_Sp'] = df_pro_carb_div[carbs].sum(axis=1)
df_pro_carb_div['sugar_Sp'] = df_pro_carb_div[sugar].sum(axis=1)

for i in protein:
  df_pro_carb_div[f'protein_p_{i}'] = df_pro_carb_div[i]/df_pro_carb_div['protein_Sp']

protein_cols = [f'protein_p_{i}' for i in protein]

# Replace 0 with NaN to avoid log(0); 0*log(0) is defined as 0
p_protein = df_pro_carb_div[protein_cols].replace(0, np.nan)
df_pro_carb_div['h_protein'] = - (p_protein * np.log(p_protein)).sum(axis=1)

for j in carbs:
  df_pro_carb_div[f'carb_p_{j}'] = df_pro_carb_div[j]/df_pro_carb_div['carb_Sp']

carb_cols = [f'carb_p_{j}' for j in carbs]
p_carb = df_pro_carb_div[carb_cols].replace(0, np.nan)
df_pro_carb_div['h_carb'] = - (p_carb * np.log(p_carb)).sum(axis=1)

for k in sugar:
  df_pro_carb_div[f'sugar_p_{k}'] = df_pro_carb_div[k]/df_pro_carb_div['sugar_Sp']

sugar_cols = [f'sugar_p_{k}' for k in sugar]
p_sugar = df_pro_carb_div[sugar_cols].replace(0, np.nan)
df_pro_carb_div['h_sugar'] = - (p_sugar * np.log(p_sugar)).sum(axis=1)

# Normalize h_protein, h_carbs, h_sugar

df_pro_carb_div['h_protein_norm'] = (df_pro_carb_div['h_protein']-df_pro_carb_div['h_protein'].min())/(df_pro_carb_div['h_protein'].max()-df_pro_carb_div['h_protein'].min())
df_pro_carb_div['h_carb_norm'] = (df_pro_carb_div['h_carb']-df_pro_carb_div['h_carb'].min())/(df_pro_carb_div['h_carb'].max()-df_pro_carb_div['h_carb'].min())
df_pro_carb_div['h_sugar_norm'] =   (df_pro_carb_div['h_sugar']-df_pro_carb_div['h_sugar'].min())/(df_pro_carb_div['h_sugar'].max()-df_pro_carb_div['h_sugar'].min())
df_pro_carb_div['avg_age_norm'] = (df_pro_carb_div['avg_age']-df_pro_carb_div['avg_age'].min())/(df_pro_carb_div['avg_age'].max()-df_pro_carb_div['avg_age'].min())

In [ ]:
# Variable Transformation, outlier treatment, model fit

#variable transformation
df_pro_carb_div['h_nutrients_weight_norm_log'] = np.log(df_pro_carb_div['h_nutrients_weight_norm'])

# h_nutrients_weight_norm_log windsorization

from scipy.stats.mstats import winsorize
df_pro_carb_div['h_nutrients_weight_norm_log_winsor'] = winsorize(df_pro_carb_div['h_nutrients_weight_norm_log'], limits=[0.05, 0.05])
df_pro_carb_div['h_nut_wght_norm_winsor'] = winsorize(df_pro_carb_div['h_nutrients_weight_norm'], limits=[0.05, 0.05])


model1 = ols('h_nut_wght_norm_winsor ~ h_protein_norm + h_carb_norm + h_sugar_norm + avg_age_norm', data=df_pro_carb_div).fit(cov_type='HC3')
print('\n')
print(f'\033[1mh_nut_wght_norm_winsor ~ h_protein_norm + h_carb_norm + h_sugar_norm + avg_age_norm\033[0m')
print(model1.summary())

In [ ]:
# Heatmaps

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='numpy.lib._function_base_impl')


# Avg Age and Nutrient diversity tertiles
df_1 = df_pro_carb_div.copy()
df_1['age_tertiles'] = pd.qcut(df_1['avg_age'],
                              q=3,                      # tertiles
                              labels=['Younger', 'Middle-Aged', 'Older']  # optional
                             )

df_1['Nutrient_diversity_tertiles'] = pd.qcut(
                                              df_1['h_nutrients_weight_norm'],
                                              q=3,                      # tertiles
                                              labels=['low', 'mid', 'high']  # optional
                                              )


# pivot table of pred_div vs avg_age tertiles with mean protein, carb, sugar diversities as values
summary_sugar_div = df_1.pivot_table(
                                      values=['h_sugar_norm'],
                                      index='age_tertiles',
                                      columns='Nutrient_diversity_tertiles',
                                      aggfunc='mean',
                                      observed=False
                                      )
summary_prot_div = df_1.pivot_table(
                                      values=['h_protein_norm'],
                                      index='age_tertiles',
                                      columns='Nutrient_diversity_tertiles',
                                      aggfunc='mean',
                                      observed=False
                                      )
summary_carb_div = df_1.pivot_table(
                                      values=['h_carb_norm'],
                                      index='age_tertiles',
                                      columns='Nutrient_diversity_tertiles',
                                      aggfunc='mean',
                                      observed=False
                                      )


# Changing multi-index column names for heatmap
if isinstance(summary_sugar_div.columns, pd.MultiIndex):
    summary_sugar_div.columns = summary_sugar_div.columns.get_level_values(-1)

if isinstance(summary_prot_div.columns, pd.MultiIndex):
    summary_prot_div.columns = summary_prot_div.columns.get_level_values(-1)

if isinstance(summary_carb_div.columns, pd.MultiIndex):
    summary_carb_div.columns = summary_carb_div.columns.get_level_values(-1)


CREAM = '#FAF7F2'
NAVY  = '#001A4D'
GRAY  = '#4A5568'
DARKGRAY = '#2D3748'

heatmap_configs = [
    (summary_prot_div,  'Protein', 'RdYlGn',   'heatmap_protein.png', True),
    (summary_carb_div,  'Carb',    'RdYlGn_r',  'heatmap_carb.png', False),
    (summary_sugar_div, 'Sugar',   'RdYlGn_r',  'heatmap_sugar.png', False),
]

for data, title, cmap, filename, show_ylabel in heatmap_configs:
    fig, ax = plt.subplots(figsize=(5, 4))
    fig.patch.set_facecolor(CREAM)
    ax.set_facecolor(CREAM)

    sns.heatmap(data, ax=ax, cmap=cmap, annot=False, fmt='.3f',
                annot_kws={'size': 16, 'color': NAVY,'weight': 'bold'},
                linewidths=0.5, linecolor=DARKGRAY)

    ax.set_title(title, color=NAVY, fontsize=18, fontweight='bold', pad=10)
    ax.set_xlabel('Nutrient Diversity Tertiles', color=GRAY, fontsize=16)
    ax.set_ylabel('Average Age Tertiles' if show_ylabel else '', color=GRAY, fontsize=16)
    ax.tick_params(colors=GRAY, labelsize=12)
    
     # Hide y tick labels for carb/sugar too if you want fully clean
    if not show_ylabel:
        ax.set_yticklabels([])

    # Custom colorbar labels
    colorbar = ax.collections[0].colorbar
    colorbar.set_ticks([colorbar.vmin, (colorbar.vmin + colorbar.vmax) / 2, colorbar.vmax])
    colorbar.set_ticklabels(['Low', 'Medium', 'High'])
    colorbar.ax.tick_params(colors=DARKGRAY, labelsize=16)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight', facecolor=CREAM)
    plt.close()
    print(f"Saved {filename}")


### Insight 3: Does income shape Nutrient Diversity?

In [ ]:
# OLS 
df_2 = df_msoa_grocery_income.copy()
df_2.rename(columns={'Net weekly income (£)': 'net_income_before_hc'}, inplace=True)
df_2['Diversity_divergence'] = df_2['h_items_norm']-df_2['h_nutrients_weight_norm']
df_2['net_income_log'] = np.log(df_2['net_income_before_hc'])

# winsorization of dependent variable
from scipy.stats.mstats import winsorize
df_2['Diversity_divergence_winsor'] = winsorize(df_2['Diversity_divergence'], limits=[0.065, 0.01])
df_2['Diversity_divergence_winsor_log'] = np.log(df_2['Diversity_divergence_winsor'])

from scipy.stats import pearsonr
corr1, _ = pearsonr(df_2['net_income_log'], df_2['Diversity_divergence_winsor_log'])
print(f'Pearson correlation: {corr1:.3f}')

model2 = ols('Diversity_divergence_winsor_log ~ net_income_log', data=df_2).fit(cov_type='HC3')
print('\n')
print(f'\033[1m Log(Diversity Divergence) ~ Log(Net Income) \033[0m')
print(model2.summary())
r2=model2.rsquared


In [ ]:
#scatter plot of Diversity_divergence vs hh_income_density
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.clf()
# plt.close('all') # Close all existing plot figures to ensure a fresh plot
fig, ax1 = plt.subplots(figsize=(14, 10)) # Create a new figure for the plot
fig.patch.set_facecolor('#FAF7F2')
ax1.set_facecolor('#FAF7F2')

#Regression line
sns.regplot(data=df_2,
            x="net_income_log",
            y="Diversity_divergence_winsor_log",
            scatter=False,
            color='#EE1C25',
            line_kws={'linewidth': 2.5, 'linestyle':'--'},
            ci=95,
            ax =ax1
            )

# Restyle confidence interval fill
ax1.collections[0].set_facecolor("#D46868")
ax1.collections[0].set_alpha(0.4)
ax1.collections[0].set_zorder(1)


#Scatter
sns.scatterplot(data=df_2, x="net_income_log", y="Diversity_divergence_winsor_log",
                legend=False,
                color = '#001A4D',
                alpha=1.0,
                s = 20,
                linewidth=0,
                ax = ax1,
                zorder=3
                )

# # add R² annotation (top-left inside axes)
# ax1.text(0.845, 0.95,
#          f"$R^2$ = {r2:.2f}",
#          transform= ax1.transAxes,
#          color='#001A4D', fontsize=12, ha='right', va='top')
# ax1.set_xlabel('Log(Net income)', color='#4A5568', fontsize=12)
# ax1.set_ylabel('Log(Diversity Divergence)', color='#4A5568',fontsize=12)
# ax1.set_title('Diversity Divergence vs Income', color='#001A4D', fontweight='bold', fontsize=12, pad=12)
# ax1.collections[0].colorbar

# Tick colours
ax1.tick_params(colors='#4A5568', labelsize = 26,width =2,length =4)

# Spine colours
for spine in ax1.spines.values():
    spine.set_edgecolor('#E8DDD0')

ax1.spines[['right', 'top']].set_visible(False)
ax1.spines[['left','bottom']].set_linewidth(2)
ax1.spines[['left','bottom']].set_color('#4A5568')

# Grid
ax1.grid(color='#F0EBE3', linewidth=0.5, linestyle='--', alpha=0.5)
ax1.set_ylim(bottom=ax1.get_ylim()[0] - 0.1)
ax1.set_xlabel("Log(Net Income)", color="#4A5568", fontsize=30)
ax1.set_ylabel("Log(Diversity Divergence)", color = "#4A5568", fontsize =30)
legend_elements = [
    Line2D([0], [0], color='#EE1C25', linewidth=2.5, linestyle='--', label='Regression Line'),
    Patch(facecolor="#D48B6B", alpha=0.3, edgecolor='none', label='95% Confidence Interval'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#001A4D', markersize=6, label='MSOA')
]


ax1.legend(handles=legend_elements,
           loc='upper left',
           bbox_to_anchor=(0.75, 1),
           framealpha=1.0,
           facecolor='#F0EBE3',  # matches ax background
           labelcolor='#001A4D',
           frameon= False,
           fontsize=30)
#plt.tight_layout()
plt.savefig('scatter_divergence.png', dpi=200, bbox_inches='tight', facecolor='#FAF7F2')
plt.show()

### Insight 1: Where are the Nutrient Diversity gaps - Choropleth analysis

In [ ]:
# Load MSOA geographical boundaries
geojson_path = DATA_DIR / (
    "MSOA_Dec_2011_Boundaries_Generalised_Clipped_BGC_EW_V3_2022_-5730664396045573288.geojson"
)

if not geojson_path.exists():
    raise FileNotFoundError(
        "The MSOA GeoJSON boundary file was not found in data/raw/. "
        "Check the filename and see data/README.md for source documentation."
    )

gpd_df = gpd.read_file(geojson_path)


In [ ]:
# create a new column to split MSOA11NM into two, The name and the number separately
gpd_df['MSOA11NMF']=gpd_df['MSOA11NM'].str.split('0').str[0]

# trim MSOA11NMF col
gpd_df['MSOA11NMF'] = gpd_df['MSOA11NMF'].str.strip()

gpd_df.head()

In [ ]:
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

# Filter for only London regions
london_msoa_gdf = gpd_df[gpd_df['MSOA11NMF'].isin(london_boroughs)].copy()
#london_msoa_gdf.shape
london_msoa_gdf['area_id']=london_msoa_gdf['MSOA11CD']
london_msoa_gdf.head()

In [ ]:
# Merge datasets
df_msoa_diversity = pd.merge(df_msoa_grocery_income,london_msoa_gdf,on='area_id')
df_msoa_diversity.head()

In [ ]:
import mapclassify

# Compute mean diversity per borough
borough_means = df_msoa_diversity.groupby('MSOA11NMF')['h_nutrients_weight_norm'].mean().reset_index()
borough_means.columns = ['borough', 'mean_diversity']

# Sort to see which are genuinely low
# print(borough_means.sort_values('mean_diversity').head(10))

print(borough_means[borough_means['mean_diversity'] <= 0.70].sort_values('mean_diversity'))

In [ ]:
from mapclassify import Quantiles
from matplotlib.patches import Patch
import matplotlib.cm as cm
import warnings

warnings.filterwarnings('ignore')

inner_names = [
    "City of London", "Camden", "Islington", "Hackney",
    "Tower Hamlets", "Southwark", "Lambeth",
    "Westminster", "Kensington and Chelsea"
]

gdf = gpd.GeoDataFrame(df_msoa_diversity)

fig, ax = plt.subplots(1, 1, figsize=(16, 12))
fig.patch.set_facecolor('#FAF7F2')
ax.set_facecolor('#FAF7F2')

# [left, bottom, width, height] as fractions of figure
ax.set_position([0.01, 0.15, 0.98, 0.80])

# Base nutrient map
gdf.plot(
    column="h_nutrients_weight_norm",
    cmap="RdYlGn",
    scheme="Quantiles",
    k=5,
    linewidth=0.1,
    edgecolor="black",
    legend=False,
    ax=ax
)

# Build manual patch legend
classifier = Quantiles(gdf["h_nutrients_weight_norm"], k=5)
bins = [gdf["h_nutrients_weight_norm"].min()] + list(classifier.bins)
cmap_obj = cm.get_cmap('RdYlGn')
colours = [cmap_obj(i / 4) for i in range(5)]

handles = [
    Patch(facecolor=colours[i],
          edgecolor='#4A5568',
          linewidth=0.8,
          label=f'{bins[i]:.2f} – {bins[i+1]:.2f}')
    for i in range(5)
]

#ax.set_title("Nutrient diversity by MSOA in London",
           #  fontsize=22, pad=10, color='#001A4D', fontweight='bold')

# Legend inside the plot — stable regardless of font size
ax.legend(handles=handles,
          loc='lower center',
          bbox_to_anchor=(0.5, -0.2),
          framealpha=0.9,
          facecolor='#FAF7F2',
          edgecolor='#4A5568',
          fontsize=30,
          ncol=5,
          title='Nutrient Diversity',
          title_fontsize=30,
          handlelength=1.5,   # ← shorter colour patches
          handleheight=1.0,   # ← less tall patches
          columnspacing=1.2,  # ← tighten gap between columns
          borderpad=0.6      # ← less internal padding in the box
          )

# Inner London boundary
inner = gdf[gdf["MSOA11NMF"].isin(inner_names)]
inner_outline = inner.dissolve()
inner_outline.boundary.plot(ax=ax, linewidth=2.0, edgecolor="black")

# Highlight specific boroughs
boroughs_to_highlight = ['Newham', 'Barking and Dagenham', 'Bexley', 'Havering']
borough_boundaries = gdf.dissolve(by='MSOA11NMF')
highlighted = borough_boundaries[borough_boundaries.index.isin(boroughs_to_highlight)]
highlighted.boundary.plot(ax=ax, color='black', linewidth=2.5)

ax.set_axis_off()
# plt.tight_layout()
# plt.subplots_adjust(bottom=0.12)
plt.savefig('choropleth.png', dpi=200, bbox_inches='tight', facecolor='#FAF7F2')
plt.show()